In [ ]:
import matplotlib
#matplotlib.use('Agg')
path_data = '../../assets/data/'
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import math
import scipy.stats as stats
plt.style.use('fivethirtyeight')

# 분류

*[David Wagner](https://people.eecs.berkeley.edu/~daw/)가 이 장의 주요 저자입니다.*

*머신러닝(Machine learning)*은 데이터에서 자동으로 패턴을 찾고 이를 사용하여 추론하거나 예측하는 기법들의 집합입니다. 여러분은 이미 머신러닝의 한 종류인 선형 회귀를 보았습니다. 이 장에서는 새로운 기법인 *분류(classification)*를 소개합니다.

분류는 과거 사례로부터 예측하는 방법을 배우는 것입니다. 우리는 정확한 예측이 무엇인지 알고 있는 몇 가지 사례를 제공받고, 이러한 사례로부터 미래에 좋은 예측을 하는 방법을 배우고자 합니다. 다음은 분류가 실제로 사용되는 몇 가지 응용 사례입니다:

- Amazon이 받는 각 주문에 대해 Amazon은 다음을 예측하고자 합니다: ***이 주문이 사기인가?*** 그들은 각 주문에 대한 일부 정보를 가지고 있습니다(예: 총 금액, 주문이 이 고객이 이전에 사용한 주소로 배송되는지 여부, 배송 주소가 신용카드 소유자의 청구 주소와 동일한지 여부). 그들은 과거 주문에 대한 많은 데이터를 가지고 있으며, 이러한 과거 주문 중 어떤 것이 사기였고 어떤 것이 아닌지 알고 있습니다. 그들은 새로운 주문이 도착할 때 이러한 새로운 주문이 사기인지 예측하는 데 도움이 되는 패턴을 학습하고자 합니다.

- 온라인 데이트 사이트는 다음을 예측하고자 합니다: ***이 두 사람이 잘 맞는가?*** 그들이 잘 어울릴까요? 그들은 과거에 고객에게 제안한 매칭에 대한 많은 데이터를 가지고 있으며, 어떤 것이 성공적이었는지에 대한 아이디어를 가지고 있습니다. 새로운 고객이 가입하면, 그들에게 누가 좋은 상대가 될지 예측하고자 합니다.

- 의사들은 알고 싶어 합니다: ***이 환자가 암에 걸렸는가?*** 일부 실험실 검사의 측정값을 기반으로, 그들은 특정 환자가 암에 걸렸는지 예측할 수 있기를 원합니다. 그들은 실험실 측정값과 최종적으로 암이 발병했는지 여부를 포함한 과거 환자에 대한 많은 데이터를 가지고 있으며, 이로부터 어떤 측정값이 암(또는 비암)의 특징인지 추론하여 미래 환자를 정확하게 진단하고자 합니다.

- 정치인들은 예측하고자 합니다: ***당신이 그들에게 투표할 것인가?*** 이것은 그들을 지지할 가능성이 있는 사람들에게 모금 활동을 집중하고, 그들에게 투표할 유권자에게 투표 독려 활동을 집중하는 데 도움이 됩니다. 공개 데이터베이스와 상업 데이터베이스는 대부분의 사람들에 대한 많은 정보를 가지고 있습니다: 예를 들어, 그들이 집을 소유하는지 임대하는지; 부유한 동네에 사는지 가난한 동네에 사는지; 그들의 관심사와 취미; 쇼핑 습관 등입니다. 그리고 정치 캠페인은 일부 유권자를 조사하여 누구에게 투표할 계획인지 알아냈으므로, 정확한 답이 알려진 일부 사례를 가지고 있습니다. 이 데이터로부터, 캠페인은 다른 모든 잠재적 유권자에 대한 예측을 하는 데 도움이 되는 패턴을 찾고자 합니다.

이 모든 것이 분류 작업입니다. 이러한 각 사례에서 예측이 예/아니오 질문이라는 것을 주목하세요 -- 우리는 이것을 *이진 분류(binary classification)*라고 부르는데, 가능한 예측이 단 두 개뿐이기 때문입니다.

분류 작업에서, 우리가 예측하고자 하는 각 개인이나 상황을 *관측값(observation)*이라고 합니다. 우리는 일반적으로 많은 관측값을 가지고 있습니다. 각 관측값은 여러 *속성(attributes)*을 가지고 있으며, 이는 알려져 있습니다(예를 들어, Amazon 주문의 총 금액이나 유권자의 연간 급여). 또한, 각 관측값은 *클래스(class)*를 가지고 있으며, 이는 우리가 관심 있는 질문에 대한 답입니다(예를 들어, 사기인지 아닌지, 또는 당신에게 투표하는지 아닌지).

Amazon이 주문이 사기인지 예측할 때, 각 주문은 하나의 관측값에 해당합니다. 각 관측값은 여러 속성을 가지고 있습니다: 주문의 총 금액, 주문이 이 고객이 이전에 사용한 주소로 배송되는지 여부 등입니다. 관측값의 클래스는 0 또는 1이며, 여기서 0은 주문이 사기가 아님을 의미하고 1은 주문이 사기임을 의미합니다. 고객이 새 주문을 할 때, 우리는 그것이 사기인지 관측하지 못하지만, 그 속성은 관측하며, 이러한 속성을 사용하여 클래스를 예측하려고 합니다.

분류는 데이터를 필요로 합니다. 패턴을 찾는 것을 포함하며, 패턴을 찾으려면 데이터가 필요합니다. 여기서 데이터 과학이 등장합니다. 특히, 우리는 *훈련 데이터(training data)*에 접근할 수 있다고 가정합니다: 각 관측값의 클래스를 아는 많은 관측값들입니다. 이러한 미리 분류된 관측값들의 모음을 훈련 세트(training set)라고도 합니다. 분류 알고리즘은 훈련 세트를 분석한 다음 분류기(classifier)를 만들어냅니다: 미래 관측값의 클래스를 예측하는 알고리즘입니다.

분류기는 유용하기 위해 완벽할 필요가 없습니다. 정확도가 100% 미만이어도 유용할 수 있습니다. 예를 들어, 온라인 데이트 사이트가 때때로 나쁜 추천을 하더라도, 괜찮습니다; 그들의 고객은 이미 잘 어울리는 사람을 찾기 전에 많은 사람을 만나야 한다고 예상합니다. 물론, 분류기가 너무 많은 오류를 만들기를 원하지 않습니다 &mdash; 하지만 매번 정확한 답을 얻을 필요는 없습니다.